# Analysis (T-20)

Two independent result trees. Run All writes both. They never share a `tables/` or `figures/` folder.

| Label | Directory | What it is |
|---|---|---|
| `judge` | `results/exp_final/` | LLM-judge plates (tables 1–3, figures 1–3) |
| `human` | `results/human_primary/` | Human-census plates (tables 1–5, figures 1–4) |

Does not recompute Wilcoxon, Holm, permutation, bootstrap, or descriptive aggregation.
A missing tree is skipped. Tests may set `SIM_RESULTS_DIR` to write one synthetic directory only.
Do not edit `results/human_validation/exp_final/frozen/`.

In [ ]:
from os import environ
from pathlib import Path

from sim.reporting import write_thesis_artifacts

TARGETS = {
    "judge": "results/exp_final",
    "human": "results/human_primary",
}


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "sim"
        ).is_dir():
            return candidate
    return here


def _resolve(root: Path, raw: str) -> Path:
    path = Path(raw)
    return path if path.is_absolute() else root / path


def regenerate(results_dir: Path) -> None:
    metrics = results_dir / "metrics.csv"
    descriptive = results_dir / "descriptive.csv"
    tests = results_dir / "tests.csv"
    if not descriptive.is_file():
        print(f"skip {results_dir}: no descriptive.csv")
        return
    write_thesis_artifacts(
        results_dir,
        metrics_path=metrics,
        descriptive_path=descriptive,
        tests_path=tests,
    )
    print(f"wrote {results_dir / 'tables'} and {results_dir / 'figures'}")


root = _repo_root()
override = environ.get("SIM_RESULTS_DIR")
if override:
    regenerate(_resolve(root, override))
else:
    for label, relative in TARGETS.items():
        print(f"{label}: {root / relative}")
        regenerate(_resolve(root, relative))

Plates are files next to the CSVs, not inlined here. Open `tables/*.csv` and `figures/*.png` under the directory printed above.